In [ ]:
#ライブラリーの呼び出し
import cv2
import numpy as np
from scipy import signal
from matplotlib import pyplot as plt
import time
import pandas as pd
import os
import glob
import re 
import session_info

In [ ]:
#カレントディレクトリ表示
print(os.getcwd())

In [ ]:
#2階層下のファイルpathを取得する
files = glob.glob(os.getcwd()+'/**/*')
#拡張子aviのファイルに絞る
subfiles = [s for s in files if re.match('.*.avi',s)]
#ファイル数
print('Total Count of Files: '+str(len(subfiles)))

#ファイルpath表示
for i, name in enumerate(subfiles):
    print(i, name)

In [ ]:
#0から順に数字を上げていく
subfile=subfiles[0]
subfile

In [ ]:
#解析者が初期設定
Hz =1 #周波数設定
scale_pixel = 88 #pixelとumとの対応関係
scale_um = 1000 #pixelとumとの対応関係
LorR ='Right' #測定するピラーの選択('Left' or 'Right')

In [ ]:
if LorR == 'Left':
    pass
elif LorR == 'Right':
    pass
else:
    print('ERROR: Input \'Left\' or \'Right\' to LorR')

In [ ]:
#結果出力フォルダの作成
subfile_rename = subfile.replace('.avi', '')
subfile_rename = subfile_rename.replace(os.getcwd(),os.getcwd()+'/output')
subfile_rename_LR = subfile_rename+'/'+ LorR
os.makedirs(subfile_rename_LR, exist_ok=False)

In [ ]:
#解析動画情報の取得
def video_info(cap):
    fps = int(cap.get(cv2.CAP_PROP_FPS))#1秒あたりの静止画枚数 
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))# 動画の横幅
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))# 動画の縦幅
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))# 総フレーム数
    print('fps:',fps,'/ width:',width,'/ height:',height,'/ total frames:',total)
    return(fps,width,height)

In [ ]:
#output動画の設定
def video_output():
    outvideo = subfile_rename_LR+"/"+ os.path.basename(subfile_rename)+"_"+os.path.basename(subfile_rename_LR) + "_movie.mp4" #出力先、ファイル名の指定
    fourcc = cv2.VideoWriter_fourcc('m', 'p', '4', 'v')# 動画保存時のfourcc設定（mp4用）
    font = cv2.FONT_HERSHEY_SIMPLEX # 経過時間表示のフォント指定
    return(outvideo,fourcc,font)

In [ ]:
#ROIの左上頂点および右下頂点座標の取得
def coordinate(ROI):
    (x,y,w,h) = tuple(map(int,ROI))
    p1 = (x, y)#左上頂点
    p2 = (x+w, y+h)#右下頂点
    return(x,p1,p2)

In [ ]:
#ROIの選択
def ROI(frame):
    ROI_initial = cv2.selectROI(frame, False)#ROIの選択
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    x,p1,p2 = coordinate(ROI_initial)
    cv2.rectangle(frame, p1, p2, (0,255,0), 3)#ROI初期位置の画像保存
    outROI=subfile_rename_LR+"/"+ os.path.basename(subfile_rename)+"_"+os.path.basename(subfile_rename_LR) + "_ROI_initial.png"
    cv2.imwrite(outROI, frame)
    return(ROI_initial,x)

In [ ]:
#ROIの追跡
def tracking():
    cap = cv2.VideoCapture(subfile) #動画読み込み 
    ret, frame = cap.read()#動画からフレーム取得
    if LorR == 'Left':#左側ピラーの測定の場合は180度回転
        frame = cv2.rotate(frame, cv2.ROTATE_180)
    else:
        pass
    fps,width,height = video_info(cap)
    outvideo,fourcc,font = video_output()
    ROI_initial,x = ROI(frame)
    video = cv2.VideoWriter(outvideo, fourcc, fps, (width, height), True)# 動画の仕様（ファイル名、fourcc, FPS, サイズ, カラー）
    tracker = cv2.TrackerMIL_create()#トラッカーの生成
    tracker.init(frame, ROI_initial)#トラッカー初期化（ROI初期位置を指定）
    count = 0
    cv2.putText(frame, "{:.1f}s".format(round(count/fps, 1)), (15, 30), font, 1, (255, 0, 255), 3)
    video.write(frame)#output動画書き込み
    x_list = []
    x_list.append(x)#トラッカー左上頂点X座標の取得、リストに追加
    while True:#2フレーム目以降、最終フレームまで無限ループ
        start = time.time()#開始時刻の取得
        ret, frame = cap.read()#動画からフレーム取得（ループが実行される毎に1フレームずつ読み込まれる）
        if LorR == 'Left':#左側ピラーの測定の場合は180度回転
            frame = cv2.rotate(frame, cv2.ROTATE_180)
        else:
            pass
        if ret==False:#動画からフレーム取得失敗している場合（最終フレームを越えて実行された）はループから抜ける
            break
        else :#動画からフレーム取得が成功していれば実行する
            success, roi = tracker.update(frame)#トラッカー位置を更新する
            x,p1,p2 = coordinate(roi)
            cv2.rectangle(frame, p1, p2, (0,255,0), 3)
            count += 1
            cv2.putText(frame, "{:.1f}s".format(round(count/fps, 1)), (15, 30), font, 1, (255, 0, 255), 3)
            video.write(frame)#output動画追加書き込み
            x_list.append(x)#トラッカー左上頂点X座標の取得、リストに追加
            sleep_secs = time.time() - start
            time.sleep(sleep_secs)#猶予時間を経て処理を実行する
            cv2.imshow("TrackerMIL", frame)
            if cv2.waitKey(1) & 0xff == 27 :#escキーを押すと途中終了できる
                break
    cap.release()
    video.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    return(x_list,fps)

In [ ]:
#ROIの追跡実行
x_list,fps = tracking()

In [ ]:
#ベースラインの決定
def baseline():
    hist, bin_edges = np.histogram(x_list, bins=40)#変数histに各階級の数、bin_edgesに各階級の端点
    base=(bin_edges[np.argmax(hist)]+bin_edges[np.argmax(hist)+1])/2 #分布の多い階級の両端点の平均をベースラインとして採用する
    fig = plt.figure(figsize=(14.0, 6.0))
    plt.rcParams["font.size"] = 20 #figのフォントサイズ
    ax= fig.add_subplot(111)
    ax.set_ylabel("frequency")
    plt.hist(x_list,bins=40,color="k")
    plt.plot(base,0,'ro')
    plt.text(base, np.argmax(hist)/20, '{x:.1f}'.format(x=base), fontsize=20,color ="r")
    fig.savefig(subfile_rename_LR+"/"+ os.path.basename(subfile_rename)+"_"+ os.path.basename(subfile_rename_LR) +"_histogram.png")   
    return(base)

In [ ]:
#ベースラインからのROIの移動距離算出
def distance(base):
    dist = base-x_list #ベースラインとトラッカーX座標リストとの差分を算出
    dist = dist*scale_um/scale_pixel #単位をピクセルからumへと変換　
    sec = np.array(range(len(x_list)))/fps#経過時間をフレーム順から秒へと変換
    fig = plt.figure(figsize=(14.0, 6.0))#移動距離の経時変化を描画
    plt.rcParams["font.size"] = 20 #figのフォントサイズ
    ax = fig.add_subplot(111)
    limy1=min(dist)-max(dist)*0.1
    limy2=max(dist)+max(dist)*0.2
    plt.ylim(limy1,limy2)
    ax.scatter(sec,dist, color="k")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("distance (um)")
    fig.savefig(subfile_rename_LR+"/"+ os.path.basename(subfile_rename)+"_"+ os.path.basename(subfile_rename_LR) +"_distance.png")
    n = 3 #移動距離を移動平均(n=3)を使って平滑化
    b = np.ones(n)/n
    smoothed=np.convolve(dist, b, mode='same')
    fig = plt.figure(figsize=(14.0, 6.0))
    plt.rcParams["font.size"] = 20 #figのフォントサイズ
    ax = fig.add_subplot(111)
    plt.ylim(limy1,limy2)
    ax.scatter(sec,smoothed, color="k")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("distance (um)")
    fig.savefig(subfile_rename_LR+"/"+ os.path.basename(subfile_rename)+"_"+ os.path.basename(subfile_rename_LR) + "_smoothed_distance.png")
    return(sec,smoothed,limy1,limy2)#経過時間、平滑化した移動距離、y軸設定値を戻り値とする

In [ ]:
#波形の立ち上がり位置の取得
def rising(smoothed):
    rising_idx=[]
    for m in range(1,len(smoothed)-3):#
        if (smoothed[m] < smoothed[m+1] and smoothed[m+1] < smoothed[m+2] and smoothed[m+2] < smoothed[m+3]
             and smoothed[m-1] >= smoothed[m]):
            rising_idx.append(m)
        else:
            pass
    return(rising_idx)

In [ ]:
#ピークの検出
def peak(rising_idx,sec,smoothed):
    peak_idx, _ = signal.find_peaks(smoothed[min(rising_idx):len(smoothed)])#最初の立ち上がり地点以降の区間におけるピークを検出
    peak_filter = smoothed[min(rising_idx):len(smoothed)][peak_idx] > max(smoothed[min(rising_idx):len(smoothed)])*0.5#最大ピーク値の半分の移動距離に満たないピークは除く
    peak_idxf = np.array(range(len(smoothed)))[min(rising_idx):len(smoothed)][peak_idx][peak_filter]  #ピーク位置のインデックスを取得  
    peaks=sec[peak_idxf]#ピーク位置の秒数を取得
    peakd=smoothed[peak_idxf]#ピーク位置の移動距離を取得
    return(peak_idxf,peaks,peakd)#ピーク位置の秒数、移動距離、インデックスを戻り値とする

In [ ]:
#立ち上がり位置からピークまでの範囲で移動距離がピークの20%となる位置およびピークからの検出
def points_coordinate(rising_idx,smoothed,peak_idxf,peaks,peakd,sec):
    points_coord={}
    for i in range(1,4):
        C_idx = min(np.array(range(len(smoothed)))[rising_idx[i]:peak_idxf[i]][smoothed[rising_idx[i]:peak_idxf[i]]-peakd[i]*0.2>0])
        sec_C=(peakd[i]*0.2-smoothed[C_idx])*(sec[C_idx-1]-sec[C_idx])/(smoothed[C_idx-1]-smoothed[C_idx])+sec[C_idx]
        if rising_idx[i]+int(fps/Hz) > len(smoothed):
            tmp= len(smoothed)
        else:
            tmp= rising_idx[i]+int(fps/Hz)
        R_idx = min(np.array(range(len(smoothed)))[peak_idxf[i]:tmp][smoothed[peak_idxf[i]:tmp]-peakd[i]*0.2<0])
        sec_R=(peakd[i]*0.2-smoothed[R_idx])*(sec[R_idx-1]-sec[R_idx])/(smoothed[R_idx-1]-smoothed[R_idx])+sec[R_idx]
        col_peak_X='peak'+str(i)+'_X'
        points_coord[col_peak_X] = float('{:.3f}'.format(peaks[i]))
        col_peak_Y='peak'+str(i)+'_Y'
        points_coord[col_peak_Y] = float('{:.3f}'.format(peakd[i]))
        col_cont_X='contraction'+str(i)+'_X'
        points_coord[col_cont_X] = float('{:.3f}'.format(sec_C))
        col_cont_Y='contraction'+str(i)+'_Y'
        points_coord[col_cont_Y] = float('{:.3f}'.format(peakd[i]*0.2))
        col_relax_X='relaxation'+str(i)+'_X'
        points_coord[col_relax_X] = float('{:.3f}'.format(sec_R))
        col_relax_Y='relaxation'+str(i)+'_Y'
        points_coord[col_relax_Y] = float('{:.3f}'.format(peakd[i]*0.2))
    return(points_coord)

In [ ]:
#波形の測定、描画
def wave_analysis():
    base = baseline()
    sec,smoothed,limy1,limy2 = distance(base)
    rising_idx=rising(smoothed)
    peak_idxf,peaks,peakd=peak(rising_idx,sec,smoothed)
    points_coord=points_coordinate(rising_idx,smoothed,peak_idxf,peaks,peakd,sec)
    print(points_coord)
    fig = plt.figure(figsize=(14.0, 6.0))
    ax= fig.add_subplot(111)
    plt.ylim(limy1,limy2)
    plt.rcParams["font.size"] = 20 #figのフォントサイズ
    plt.plot(sec,smoothed, color="k")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("distance (um)")
    result = {}
    result_CT = {}
    result_RT = {}
    for i in range(1,4):
        col_peak_X='peak'+str(i)+'_X'
        points_coord[col_peak_X]
        col_peak_Y='peak'+str(i)+'_Y'
        points_coord[col_peak_Y]
        plt.scatter(points_coord[col_peak_X],points_coord[col_peak_Y],facecolor='None', edgecolors='red')
        plt.vlines(points_coord[col_peak_X],0,points_coord[col_peak_Y],'r')
        col_cont_X='contraction'+str(i)+'_X'
        points_coord[col_cont_X]
        col_cont_Y='contraction'+str(i)+'_Y'
        points_coord[col_cont_Y]
        plt.scatter(points_coord[col_cont_X],points_coord[col_cont_Y], facecolor='None', edgecolors='red')
        col_relax_X='relaxation'+str(i)+'_X'
        points_coord[col_relax_X]
        col_relax_Y='relaxation'+str(i)+'_Y'
        points_coord[col_relax_Y]
        plt.scatter(points_coord[col_relax_X],points_coord[col_relax_Y],facecolor='None', edgecolors='red')
        plt.hlines(points_coord[col_relax_Y],points_coord[col_cont_X],points_coord[col_relax_X],colors="r")
        col_distance ="peak"+str(i)+"_d"
        result[col_distance] = float("{:.3f}".format(points_coord[col_peak_Y]))
        CT="peak"+str(i)+"_CT"
        result_CT[CT] = float("{:.3f}".format(points_coord[col_peak_X]-points_coord[col_cont_X]))
        RT="peak"+str(i)+"_RT"
        result_RT[RT] =float("{:.3f}".format(points_coord[col_relax_X]-points_coord[col_peak_X]))
    fig.savefig(subfile_rename_LR+"/"+ os.path.basename(subfile_rename)+"_"+ os.path.basename(subfile_rename_LR) + "_wave.png")
    mean={}
    mean_d=sum(result.values())/len(result.values())
    mean_CT=sum(result_CT.values())/len(result_CT.values())
    mean_RT=sum(result_RT.values())/len(result_RT.values())
    mean["mean_d"]=float("{:.3f}".format(mean_d))
    mean["mean_CT"]=float("{:.3f}".format(mean_CT))
    mean["mean_RT"]=float("{:.3f}".format(mean_RT))
    result["LR"] = LorR
    result["filename"] = subfile
    result.update(result_CT)
    result.update(result_RT)
    result.update(mean)
    result=pd.DataFrame(result,index=[0])
    cols = ["LR","filename","peak1_d","peak2_d","peak3_d","peak1_CT","peak2_CT","peak3_CT","peak1_RT","peak2_RT","peak3_RT",
            "mean_d","mean_CT","mean_RT"]
    df = pd.DataFrame(index=[], columns=cols)
    df = pd.concat([df, result])
    df.to_csv(subfile_rename_LR+"/"+ os.path.basename(subfile_rename)+"_"+ os.path.basename(subfile_rename_LR) +".csv",
             index=False) 

In [ ]:
wave_analysis()

In [ ]:
session_info.show()